## OLD -> DOES NOT WORK ANYMORE : @PATRICK! .env gibts nicht mehr


`.env` must contain the following values:
```
SERVER_IP=your_server_ip
FTP_PORT=21
FTP_USER=your_username
FTP_PASSWORD=your_password
```

Use the same `.venv` you use for dev here with python >= 3.12.10

## Scrapping

### Getting match_ids from local db to minimize downloads (sync with db first)

In [1]:
from pathlib import Path
import sqlite3

# --- PATHS ---


BASE_DIR = Path().resolve().parents[2] / "Internomat"
DB_FILE = BASE_DIR / "internomat.db"
DEMO_DIR = BASE_DIR / "test_demos"

print("Using DB:", DB_FILE)


# --- DB ---


def get_conn_local():
    conn = sqlite3.connect(DB_FILE, timeout=10)
    conn.row_factory = sqlite3.Row
    return conn

def get_all_matches_with_maps_local():
    with get_conn_local() as conn:
        rows = conn.execute("""
        SELECT 
            m.match_id,
            m.team1_name,
            m.team2_name,
            mm.map_number,
            mm.map_name
        FROM matches m
        LEFT JOIN match_maps mm 
            ON m.match_id = mm.match_id
        ORDER BY m.match_id, mm.map_number
        """).fetchall()

    matches = {}

    for row in rows:
        match_id = int(row["match_id"])

        if match_id not in matches:
            matches[match_id] = {
                "match_id": match_id,
                "team1": row["team1_name"],
                "team2": row["team2_name"],
                "maps": []
            }

        if row["map_number"] is not None:
            matches[match_id]["maps"].append({
                "map_number": int(row["map_number"]),
                "map_name": row["map_name"]
            })

    return list(matches.values())


# --- SHARED UTILS ---


def extract_match_id(filename):
    try:
        return int(filename.split("_")[2])
    except:
        return None

def build_match_id_set(matches):
    return set(m["match_id"] for m in matches)


# --- LOAD MATCHES ---


matches = get_all_matches_with_maps_local()
valid_match_ids = build_match_id_set(matches)

print(f"\nLoaded {len(matches)} matches")
print(f"Valid match_ids: {sorted(valid_match_ids)}")

Using DB: D:\Git_repo\Internomat\internomat.db

Loaded 6 matches
Valid match_ids: [-1735857110, -1160320022, 1, 2, 3, 4]



### Getting all .dem from server (for now)

In [5]:
from ftplib import FTP
from dotenv import load_dotenv
import os
from tqdm import tqdm
import zlib

# --- ENV / CFG ---

env_path = BASE_DIR / ".env"
load_dotenv(env_path)

cfg_path = BASE_DIR / "internomat_settings.cfg"

def read_cfg(path):
    data = {}
    if not path.exists():
        return data

    with open(path, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()
            if not line or line.startswith("#") or line.startswith(";"):
                continue
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            data[key.strip()] = value.strip()

    return data

cfg = read_cfg(cfg_path)

# Prefer .cfg settings, fallback to .env values
FTP_HOST = cfg.get("demo_ftp_host") or os.getenv("SERVER_IP")
FTP_PORT = int(cfg.get("demo_ftp_port") or os.getenv("FTP_PORT", 21))
FTP_USER = cfg.get("demo_ftp_user") or os.getenv("FTP_USER")
FTP_PASSWORD = cfg.get("demo_ftp_password") or os.getenv("FTP_PASSWORD")
REMOTE_DIR = cfg.get("demo_remote_path") or "/cs2/game/csgo/MatchZy"
DEMO_DIR.mkdir(parents=True, exist_ok=True)

FTP_READY = bool(FTP_HOST and FTP_USER and FTP_PASSWORD)
if not FTP_READY:
    print("[WARN] Missing FTP settings in .cfg/.env. Skipping download step.")


# --- HELPERS ---

def extract_parts(filename):
    # 2026-03-20_22-48-10_4_cs_office_team_...vs_...
    parts = filename.replace(".dem", "").split("_")

    try:
        date = parts[0]
        time = parts[1]
        raw_match_id = parts[2]
        map_name = f"{parts[3]}_{parts[4]}"
        return date, time, raw_match_id, map_name
    except:
        return None, None, None, None


def to_int_or_none(value):
    try:
        if value is None:
            return None
        return int(str(value).strip())
    except Exception:
        return None


def build_recovery_match_id(date, time, raw_match_id, filename):
    seed = f"{date}_{time}_{raw_match_id}_{filename}"
    return -1 - (zlib.crc32(seed.encode("utf-8")) & 0x7FFFFFFF)


def build_recovery_map_number(match_id, map_name, filename):
    seed = f"{match_id}_{map_name}_{filename}"
    return 1 + (zlib.crc32(seed.encode("utf-8")) & 0x7FFFFFFF)


def get_map_number(match, map_name):
    for m in match.get("maps", []):
        if m["map_name"] == map_name:
            return m["map_number"]
    return None


def build_match_lookup(matches):
    return {m["match_id"]: m for m in matches}


def normalize_demo_identity(remote_filename, match_lookup):
    date, time, raw_match_id, map_name = extract_parts(remote_filename)
    if not date or not time or not map_name:
        return None

    parsed_match_id = to_int_or_none(raw_match_id)
    normalized_match_id = (
        parsed_match_id
        if parsed_match_id is not None and parsed_match_id > 0
        else build_recovery_match_id(date, time, raw_match_id, remote_filename)
    )

    map_number = None
    # Optional DB assist only when available; no filtering by DB IDs.
    if parsed_match_id is not None and parsed_match_id in match_lookup:
        map_number = get_map_number(match_lookup[parsed_match_id], map_name)

    recovered_from_catalog_miss = False
    if map_number is None:
        map_number = build_recovery_map_number(
            match_id=normalized_match_id,
            map_name=map_name,
            filename=remote_filename,
        )
        recovered_from_catalog_miss = True

    normalized_name = f"{date}_{time}_match_{normalized_match_id}_map_{int(map_number)}_{map_name}.dem"
    return {
        "date": date,
        "time": time,
        "raw_match_id": raw_match_id,
        "match_id": normalized_match_id,
        "map_name": map_name,
        "map_number": int(map_number),
        "normalized_name": normalized_name,
        "recovered_from_catalog_miss": recovered_from_catalog_miss,
    }


# --- FTP ---
def download_demo():
    if not FTP_READY:
        return

    print(f"\n[FTP] Connecting to {FTP_HOST}:{FTP_PORT}...")

    ftp = FTP()

    try:
        ftp.connect(FTP_HOST, FTP_PORT, timeout=10)
        ftp.login(FTP_USER, FTP_PASSWORD)
        ftp.set_pasv(True)

        print("[FTP] Connected")

        ftp.cwd(REMOTE_DIR)
        files = ftp.nlst()

        downloaded, skipped, ignored = 0, 0, 0
        match_lookup = build_match_lookup(matches)

        print(f"[FTP] Found {len(files)} files")

        for file in files:
            if not file.endswith(".dem"):
                continue

            identity = normalize_demo_identity(file, match_lookup)
            if not identity:
                ignored += 1
                continue

            normalized_name = identity["normalized_name"]
            local_file = DEMO_DIR / normalized_name

            if local_file.exists():
                print(f"[SKIP] {normalized_name}")
                skipped += 1
                continue

            print(f"[DOWNLOAD] {file}")
            print(f"           -> {normalized_name}")
            if identity["recovered_from_catalog_miss"]:
                print("           -> using recovered map_number (DB map lookup miss)")

            try:
                filesize = ftp.size(file)

                with open(local_file, "wb") as f, tqdm(
                    total=filesize,
                    unit="B",
                    unit_scale=True,
                    desc=normalized_name,
                    leave=False,
                ) as pbar:

                    def callback(data):
                        f.write(data)
                        pbar.update(len(data))

                    ftp.retrbinary(f"RETR {file}", callback)

                downloaded += 1

            except Exception as e:
                print(f"[ERROR] {file}: {e}")

        print("\n[FTP] Done")
        print(f"Downloaded: {downloaded}")
        print(f"Skipped: {skipped}")
        print(f"Ignored: {ignored}")

    finally:
        try:
            ftp.quit()
        except:
            pass

# --- call ---
download_demo()


[FTP] Connecting to 83.141.26.49:21...
[FTP] Connected
[FTP] Found 8 files
[DOWNLOAD] 2026-03-19_21-10-26_-1_de_thera_team_Captain_Blake_vs_team_Tundra_ツ.dem
           -> 2026-03-19_21-10-26_match_-510014804_map_1652435401_de_thera.dem
           -> using recovered map_number (DB map lookup miss)


[SKIP] 2026-03-19_21-55-45_match_1_map_0_de_warden.dem
[SKIP] 2026-03-20_20-40-16_match_2_map_0_de_thera.dem
[SKIP] 2026-03-20_21-57-48_match_3_map_0_de_mills.dem
[SKIP] 2026-03-20_22-48-10_match_4_map_0_cs_office.dem
[DOWNLOAD] 2026-03-27_20-33-52_-1_de_dust2_team_Shouzen_vs_team_Dottersack.dem
           -> 2026-03-27_20-33-52_match_-1160320022_map_906937082_de_dust2.dem
           -> using recovered map_number (DB map lookup miss)


[DOWNLOAD] 2026-03-27_21-10-25_-1_de_cbble_d_team_Couchyyy1337_vs_team_KeTaNoS.dem
           -> 2026-03-27_21-10-25_match_-1735857110_map_1431889500_de_cbble.dem
           -> using recovered map_number (DB map lookup miss)


[DOWNLOAD] 2026-03-27_21-51-13_-1_de_thera_team_Tundra_ツ_vs_team_KeTaNoS.dem
           -> 2026-03-27_21-51-13_match_-1883972372_map_989206043_de_thera.dem
           -> using recovered map_number (DB map lookup miss)



[FTP] Done
Downloaded: 4
Skipped: 4
Ignored: 0


### Match demo files to local matches

In [6]:
from pathlib import Path
from awpy.demo import Demo
import pandas as pd
import polars as pl


DEMO_DIR = Path().resolve().parents[2] / "Internomat" / "test_demos"


# --- HELPERS ---

def extract_ids_from_normalized(filename):
    # 2026-03-20_22-48-10_match_4_map_1_cs_office.dem
    parts = filename.replace(".dem", "").split("_")

    try:
        # find dynamic positions (safer than fixed indices)
        match_idx = parts.index("match")
        map_idx = parts.index("map")

        match_id = int(parts[match_idx + 1])
        map_number = int(parts[map_idx + 1])

        return match_id, map_number

    except Exception:
        return None, None


# --- MATCH + LOAD DEMOS ---

def match_and_load_demos():
    demo_files = list(DEMO_DIR.glob("*.dem"))

    print(f"\n[Matcher] Found {len(demo_files)} normalized demos\n")

    results = []
    failed = []

    for file in demo_files:
        match_id, map_number = extract_ids_from_normalized(file.name)

        if match_id is None or map_number is None:
            print(f"[SKIP] Invalid normalized file: {file.name}")
            continue

        print(f"[LOAD] {file.name} (match={match_id}, map={map_number})")

        try:
            demo = Demo(str(file))
            demo.parse()

            results.append({
                "file": file,
                "match_id": match_id,
                "map_number": map_number,
                "demo": demo
            })

        except Exception as e:
            print(f"[FAILED] {file.name}: {e}")
            failed.append(file.name)

    print(f"\n[Matcher] Loaded {len(results)} demos successfully")
    print(f"[Matcher] Failed: {len(failed)}")

    if failed:
        print("\nFailed demos:")
        for f in failed:
            print(f)

    return results


# --- RUN ---

matched = match_and_load_demos()


[Matcher] Found 8 normalized demos

[LOAD] 2026-03-19_21-10-26_match_-510014804_map_1652435401_de_thera.dem (match=-510014804, map=1652435401)
[FAILED] 2026-03-19_21-10-26_match_-510014804_map_1652435401_de_thera.dem: EntityNotFound
[LOAD] 2026-03-19_21-55-45_match_1_map_0_de_warden.dem (match=1, map=0)
[LOAD] 2026-03-20_20-40-16_match_2_map_0_de_thera.dem (match=2, map=0)
[FAILED] 2026-03-20_20-40-16_match_2_map_0_de_thera.dem: IllegalPathOp
[LOAD] 2026-03-20_21-57-48_match_3_map_0_de_mills.dem (match=3, map=0)
[LOAD] 2026-03-20_22-48-10_match_4_map_0_cs_office.dem (match=4, map=0)
[LOAD] 2026-03-27_20-33-52_match_-1160320022_map_906937082_de_dust2.dem (match=-1160320022, map=906937082)
[LOAD] 2026-03-27_21-10-25_match_-1735857110_map_1431889500_de_cbble.dem (match=-1735857110, map=1431889500)
[LOAD] 2026-03-27_21-51-13_match_-1883972372_map_989206043_de_thera.dem (match=-1883972372, map=989206043)
[FAILED] 2026-03-27_21-51-13_match_-1883972372_map_989206043_de_thera.dem: IllegalPath

### Parser

In [7]:
# --- PARSER ---

def parse_demo_full(demo):
    data = {}

    data["header"] = demo.header

    table_map = {
        "rounds": "rounds",
        "kills": "kills",
        "damages": "damages",
        "grenades": "grenades",
        "shots": "shots",
        "footsteps": "footsteps",
        "smokes": "smokes",
        "infernos": "infernos",
        "bomb": "bomb",
        "ticks": "ticks",
        "player_round_totals": "player_round_totals",
        "server_cvars": "server_cvars",
    }

    for key, attr in table_map.items():
        try:
            value = getattr(demo, attr, None)

            if value is None:
                data[key] = None
                continue

            if isinstance(value, pd.DataFrame):
                df = value
            elif isinstance(value, list):
                df = pd.DataFrame(value)
            else:
                df = value

            data[key] = df

        except Exception:
            data[key] = None

    return data

# --- HELPERS ---

def print_headers(data, preview=False):
    print("\n=== TABLE HEADERS ===\n")

    for key, value in data.items():
        print(f"[TABLE] {key}")

        if value is None:
            print("  -> None\n")
            continue

        if isinstance(value, pd.DataFrame):
            print(f"  Rows: {len(value)}")
            print(f"  Columns ({len(value.columns)}):")
            for col in value.columns:
                print(f"    - {col}")

            if preview:
                print("\n  Preview:")
                print(value.head(3))

        elif isinstance(value, pl.DataFrame):
            print(f"  Rows: {value.height}")
            print(f"  Columns ({len(value.columns)}):")
            for col in value.columns:
                print(f"    - {col}")

        else:
            print(f"  Type: {type(value).__name__}")

        print("")

def get_cvars(data):
    cvars = data.get("server_cvars")
    if cvars is None:
        print("[WARN] No CVARs")
    return cvars

# --- BUILD DEMO DICT ---

def build_demo_data(matched):
    demo_dict = {}
    failed = 0

    print("\n[Parser] Building structured datasets...\n")

    for entry in matched:
        match_id = entry["match_id"]
        map_number = entry["map_number"]
        demo = entry["demo"]
        filename = entry["file"].name

        key = (match_id, map_number)

        print(f"[PARSE] {filename} → match={match_id}, map={map_number}")

        try:
            data = parse_demo_full(demo)

            demo_dict[key] = data

        except Exception as e:
            print(f"[FAILED] {filename}: {e}")
            failed += 1

    print("\n[Parser] Done")
    print(f"Parsed: {len(demo_dict)}")
    print(f"Failed: {failed}")

    return demo_dict

# --- RUN ---

demo_data = build_demo_data(matched)


# --- OPTIONAL INSPECTION (COMPACT) ---

# print compact header for each demo / dataset included
if demo_data:
    print("\n=== DEMO HEADERS (COMPACT) ===\n")
    for (match_id, map_number), sample_data in sorted(demo_data.items()):
        header = sample_data.get("header", {})
        map_name = header.get("map_name", "?")
        tick_count = header.get("tick_count", "?")
        print(f"[M{match_id}|Map{map_number}] {map_name:15} | Ticks: {tick_count}")



[Parser] Building structured datasets...

[PARSE] 2026-03-19_21-55-45_match_1_map_0_de_warden.dem → match=1, map=0
[PARSE] 2026-03-20_21-57-48_match_3_map_0_de_mills.dem → match=3, map=0
[PARSE] 2026-03-20_22-48-10_match_4_map_0_cs_office.dem → match=4, map=0
[PARSE] 2026-03-27_20-33-52_match_-1160320022_map_906937082_de_dust2.dem → match=-1160320022, map=906937082
[PARSE] 2026-03-27_21-10-25_match_-1735857110_map_1431889500_de_cbble.dem → match=-1735857110, map=1431889500

[Parser] Done
Parsed: 5
Failed: 0

=== DEMO HEADERS (COMPACT) ===

[M-1735857110|Map1431889500] de_cbble_d      | Ticks: ?
[M-1160320022|Map906937082] de_dust2        | Ticks: ?
[M1|Map0] de_warden       | Ticks: ?
[M3|Map0] de_mills        | Ticks: ?
[M4|Map0] cs_office       | Ticks: ?


## Demo to DB Middleware

In [8]:
# --- LOAD CACHED PARSED PAYLOAD (.pkl) AND PRINT TABLE HEADERS ---

import sys
from pathlib import Path
import pandas as pd
import polars as pl

# Make src importable in notebook context
sys.path.insert(0, str(BASE_DIR / "src"))

from services import demo_cache


def print_payload_table_headers(payload):
    """Print headers (columns/keys) for each stored table in cached payload."""
    if not isinstance(payload, dict):
        print("Payload is not a dict.")
        return

    print("\n=== STORED TABLE HEADERS ===")
    for table_name, table_data in payload.items():
        print(f"\n[TABLE] {table_name}")

        if table_data is None:
            print("  -> None")
            continue

        if isinstance(table_data, pd.DataFrame):
            print(f"  Type: pandas | Rows: {len(table_data)} | Cols: {len(table_data.columns)}")
            print(f"  Columns: {list(table_data.columns)}")
            continue

        if isinstance(table_data, pl.DataFrame):
            print(f"  Type: polars | Rows: {table_data.height} | Cols: {len(table_data.columns)}")
            print(f"  Columns: {list(table_data.columns)}")
            continue

        if isinstance(table_data, dict):
            keys = list(table_data.keys())
            print(f"  Type: dict | Keys: {len(keys)}")
            print(f"  Keys: {keys}")
            continue

        if isinstance(table_data, list):
            print(f"  Type: list | Items: {len(table_data)}")
            if table_data and isinstance(table_data[0], dict):
                print(f"  First item keys: {list(table_data[0].keys())}")
            continue

        print(f"  Type: {type(table_data).__name__}")


def load_latest_cached_payload(parsed_cache_dir):
    """Load the latest parsed cache entry via service and return payload + manifest."""
    entries = demo_cache.list_cached_demos(parsed_cache_dir)
    if not entries:
        print(f"No cached demos found in {parsed_cache_dir}")
        return None, None

    latest = entries[0]  # sorted by updated_at desc in service
    match_id = latest.get("match_id")
    map_number = latest.get("map_number")

    payload = demo_cache.load_parsed_demo(
        cache_dir=parsed_cache_dir,
        match_id=match_id,
        map_number=map_number,
    )

    if payload is None:
        print(f"Failed to load payload for match={match_id}, map={map_number}")
        return None, latest

    print("Loaded cached payload:")
    print(f"  match_id={match_id}, map_number={map_number}")
    print(f"  filename={latest.get('filename')}")
    print(f"  updated_at={latest.get('updated_at')}")

    return payload, latest


# Default cache location used by app: <repo>/demos/parsed
PARSED_CACHE_DIR = BASE_DIR / "demos" / "parsed"

payload, manifest = load_latest_cached_payload(PARSED_CACHE_DIR)
if payload is not None:
    print_payload_table_headers(payload)

[00:10:09.889] [DEBUG] [CACHE] Loaded parsed demo match=-1735857110 map=1431889500
Loaded cached payload:
  match_id=-1735857110, map_number=1431889500
  filename=2026-03-27_21-10-25_match_-1735857110_map_1431889500_de_cbble.pkl
  updated_at=2026-03-27T22:37:32.297841

=== STORED TABLE HEADERS ===

[TABLE] header
  Type: dict | Keys: 12
  Keys: ['demo_file_stamp', 'addons', 'map_name', 'server_name', 'fullpackets_version', 'demo_version_guid', 'allow_clientside_entities', 'allow_clientside_particles', 'demo_version_name', 'client_name', 'patch_version', 'game_directory']

[TABLE] events
  Type: dict | Keys: 20
  Keys: ['player_death', 'hegrenade_detonate', 'smokegrenade_expired', 'flashbang_detonate', 'item_pickup', 'bomb_pickup', 'player_sound', 'inferno_expire', 'inferno_startburn', 'weapon_fire', 'player_hurt', 'round_officially_ended', 'bomb_defused', 'player_spawn', 'smokegrenade_detonate', 'bomb_dropped', 'round_freeze_end', 'bomb_planted', 'round_start', 'round_end']

[TABLE] ro

In [15]:
# --- FINAL SCOREBOARD PER TEAM (full fields; side-switch aware; ALL parsed matches) ---

from collections import Counter, defaultdict
import pandas as pd
import polars as pl


UTILITY_TOKENS = {
    "hegrenade", "inferno", "molotov", "incgrenade", "flashbang", "decoy", "smokegrenade",
}


def to_polars(df_like):
    if df_like is None:
        return None
    if isinstance(df_like, pl.DataFrame):
        return df_like
    if isinstance(df_like, pd.DataFrame):
        return pl.from_pandas(df_like)
    return None


def normalize_side(value):
    txt = str(value or "").strip().upper()
    if not txt:
        return None
    if txt in {"CT", "CT_SIDE"} or "COUNTER" in txt:
        return "CT"
    if txt in {"T", "T_SIDE"} or "TERROR" in txt:
        return "T"
    return None


def is_generic_side_label(value):
    txt = str(value or "").strip().upper().replace(" ", "")
    generic = {
        "", "CT", "CT_SIDE", "T", "T_SIDE",
        "COUNTER", "COUNTERTERRORIST", "COUNTER-TERRORIST", "COUNTER-TERRORISTS",
        "COUNTER_TERRORIST", "COUNTER_TERRORISTS", "TERRORIST", "TERRORISTS",
    }
    return txt in generic


def normalize_player_id(value):
    if value is None:
        return None
    try:
        sid = int(value)
        if sid <= 0:
            return None
        return sid
    except Exception:
        return None


def safe_int(value, default=0):
    try:
        if value is None:
            return default
        return int(value)
    except Exception:
        return default


def detect_column(df, candidates):
    if df is None:
        return None
    cols = set(df.columns)
    for name in candidates:
        if name in cols:
            return name
    return None


def get_damage_column(damages):
    return detect_column(damages, ["dmg_health_real", "dmg_health", "damage", "hp_damage"])


def is_utility_damage_row(row):
    token_values = []
    for key in ["weapon", "weapon_name", "weapon_class", "weapon_type", "weapon_item"]:
        if key in row and row.get(key) is not None:
            token_values.append(str(row.get(key)).strip().lower())

    joined = " ".join(token_values)
    if not joined:
        return False

    return any(tok in joined for tok in UTILITY_TOKENS)


def collect_round_side_members(payload):
    round_side = defaultdict(lambda: {"CT": set(), "T": set()})

    def add_player(round_num, side_value, steamid):
        side = normalize_side(side_value)
        sid = normalize_player_id(steamid)
        if side is None or sid is None or round_num is None:
            return

        r = safe_int(round_num, default=0)
        if r <= 0:
            return

        round_side[r][side].add(sid)

    kills = to_polars(payload.get("kills"))
    if kills is not None and kills.height > 0:
        cols = set(kills.columns)
        needed = [c for c in [
            "round_num",
            "attacker_steamid", "attacker_side",
            "victim_steamid", "victim_side",
            "assister_steamid", "assister_side",
        ] if c in cols]

        for row in kills.select(needed).to_dicts():
            rn = row.get("round_num")
            add_player(rn, row.get("attacker_side"), row.get("attacker_steamid"))
            add_player(rn, row.get("victim_side"), row.get("victim_steamid"))
            add_player(rn, row.get("assister_side"), row.get("assister_steamid"))

    damages = to_polars(payload.get("damages"))
    if damages is not None and damages.height > 0:
        cols = set(damages.columns)
        needed = [c for c in [
            "round_num",
            "attacker_steamid", "attacker_side",
            "victim_steamid", "victim_side",
        ] if c in cols]

        for row in damages.select(needed).to_dicts():
            rn = row.get("round_num")
            add_player(rn, row.get("attacker_side"), row.get("attacker_steamid"))
            add_player(rn, row.get("victim_side"), row.get("victim_steamid"))

    shots = to_polars(payload.get("shots"))
    if shots is not None and shots.height > 0:
        cols = set(shots.columns)
        needed = [c for c in ["round_num", "player_steamid", "player_side"] if c in cols]
        if {"round_num", "player_steamid", "player_side"}.issubset(set(needed)):
            for row in shots.select(needed).to_dicts():
                add_player(row.get("round_num"), row.get("player_side"), row.get("player_steamid"))

    ticks = to_polars(payload.get("ticks"))
    if ticks is not None and ticks.height > 0:
        cols = set(ticks.columns)
        needed = [c for c in ["round_num", "steamid", "side"] if c in cols]
        if {"round_num", "steamid", "side"}.issubset(set(needed)):
            for row in ticks.select(needed).unique().to_dicts():
                add_player(row.get("round_num"), row.get("side"), row.get("steamid"))

    return round_side


def collect_explicit_round_side_names(payload):
    names = defaultdict(lambda: {"CT": [], "T": []})

    for table_key in ["kills", "damages", "shots", "footsteps", "ticks"]:
        df = to_polars(payload.get(table_key))
        if df is None or df.height == 0:
            continue

        cols = set(df.columns)
        needed = [c for c in ["round_num", "ct_side", "t_side"] if c in cols]
        if "round_num" not in needed:
            continue

        for row in df.select(needed).to_dicts():
            rn = row.get("round_num")
            if rn is None:
                continue
            r = safe_int(rn, default=0)
            if r <= 0:
                continue

            ct_val = row.get("ct_side") if "ct_side" in row else None
            t_val = row.get("t_side") if "t_side" in row else None

            if ct_val is not None and not is_generic_side_label(ct_val):
                names[r]["CT"].append(str(ct_val).strip())
            if t_val is not None and not is_generic_side_label(t_val):
                names[r]["T"].append(str(t_val).strip())

    return names


def infer_team_assignment(round_side):
    rounds = sorted(round_side.keys())
    if not rounds:
        return {}

    team_member_counts = {"TeamA": Counter(), "TeamB": Counter()}
    assignment = {}

    first = rounds[0]
    assignment[first] = {"CT": "TeamA", "T": "TeamB"}
    for sid in round_side[first]["CT"]:
        team_member_counts["TeamA"][sid] += 1
    for sid in round_side[first]["T"]:
        team_member_counts["TeamB"][sid] += 1

    for r in rounds[1:]:
        ct_set = round_side[r]["CT"]
        t_set = round_side[r]["T"]

        score_normal = (
            sum(team_member_counts["TeamA"][sid] for sid in ct_set)
            + sum(team_member_counts["TeamB"][sid] for sid in t_set)
        )
        score_swapped = (
            sum(team_member_counts["TeamB"][sid] for sid in ct_set)
            + sum(team_member_counts["TeamA"][sid] for sid in t_set)
        )

        if score_swapped > score_normal:
            assignment[r] = {"CT": "TeamB", "T": "TeamA"}
            ct_team, t_team = "TeamB", "TeamA"
        else:
            assignment[r] = {"CT": "TeamA", "T": "TeamB"}
            ct_team, t_team = "TeamA", "TeamB"

        for sid in ct_set:
            team_member_counts[ct_team][sid] += 1
        for sid in t_set:
            team_member_counts[t_team][sid] += 1

    return assignment


def infer_team_display_names(round_assignment, explicit_round_side_names):
    team_name_votes = {"TeamA": Counter(), "TeamB": Counter()}

    for r, side_to_team in round_assignment.items():
        for side in ["CT", "T"]:
            team_key = side_to_team[side]
            for name in explicit_round_side_names[r][side]:
                team_name_votes[team_key][name] += 1

    team_labels = {}
    for team_key in ["TeamA", "TeamB"]:
        if team_name_votes[team_key]:
            team_labels[team_key] = team_name_votes[team_key].most_common(1)[0][0]
        else:
            team_labels[team_key] = team_key

    return team_labels


def build_player_name_lookup(payload):
    names = {}

    players = to_polars(payload.get("player_round_totals"))
    if players is not None and players.height > 0:
        cols = set(players.columns)
        if {"steamid", "name"}.issubset(cols):
            for row in players.select(["steamid", "name"]).to_dicts():
                sid = normalize_player_id(row.get("steamid"))
                name = row.get("name")
                if sid is not None and name:
                    names[sid] = str(name)

    kills = to_polars(payload.get("kills"))
    if kills is not None and kills.height > 0:
        cols = set(kills.columns)
        needed = [c for c in [
            "attacker_steamid", "attacker_name",
            "victim_steamid", "victim_name",
            "assister_steamid", "assister_name",
        ] if c in cols]

        for row in kills.select(needed).to_dicts():
            for sid_key, name_key in [
                ("attacker_steamid", "attacker_name"),
                ("victim_steamid", "victim_name"),
                ("assister_steamid", "assister_name"),
            ]:
                if sid_key not in row or name_key not in row:
                    continue
                sid = normalize_player_id(row.get(sid_key))
                name = row.get(name_key)
                if sid is not None and name and sid not in names:
                    names[sid] = str(name)

    return names


def build_player_rounds_lookup(payload):
    rounds_lookup = Counter()

    players = to_polars(payload.get("player_round_totals"))
    if players is not None and players.height > 0:
        cols = set(players.columns)
        if "steamid" in cols and "n_rounds" in cols:
            for row in players.select(["steamid", "n_rounds"]).to_dicts():
                sid = normalize_player_id(row.get("steamid"))
                if sid is None:
                    continue
                n = safe_int(row.get("n_rounds"), default=0)
                if n > rounds_lookup[sid]:
                    rounds_lookup[sid] = n

    if rounds_lookup:
        return rounds_lookup

    # fallback: derive from unique rounds player appeared in
    round_sets = defaultdict(set)
    for table_key, sid_col in [
        ("kills", "attacker_steamid"),
        ("kills", "victim_steamid"),
        ("kills", "assister_steamid"),
        ("damages", "attacker_steamid"),
        ("damages", "victim_steamid"),
        ("shots", "player_steamid"),
        ("ticks", "steamid"),
    ]:
        df = to_polars(payload.get(table_key))
        if df is None or df.height == 0:
            continue

        cols = set(df.columns)
        if sid_col not in cols or "round_num" not in cols:
            continue

        for row in df.select([sid_col, "round_num"]).to_dicts():
            sid = normalize_player_id(row.get(sid_col))
            rn = safe_int(row.get("round_num"), default=0)
            if sid is not None and rn > 0:
                round_sets[sid].add(rn)

    for sid, rset in round_sets.items():
        rounds_lookup[sid] = len(rset)

    return rounds_lookup


def get_round_winner_side_map(payload):
    winner_map = {}
    rounds = to_polars(payload.get("rounds"))
    if rounds is None or rounds.height == 0:
        return winner_map

    winner_col = detect_column(rounds, ["winner_side", "winning_side", "round_winner_side", "winning_team"])
    if winner_col is None:
        return winner_map

    cols = set(rounds.columns)
    if "round_num" not in cols:
        return winner_map

    for row in rounds.select(["round_num", winner_col]).to_dicts():
        rn = safe_int(row.get("round_num"), default=0)
        side = normalize_side(row.get(winner_col))
        if rn > 0 and side is not None:
            winner_map[rn] = side

    return winner_map


def compute_entry_stats(payload):
    attempts = Counter()
    wins = Counter()

    kills = to_polars(payload.get("kills"))
    if kills is None or kills.height == 0:
        return attempts, wins

    winner_side_by_round = get_round_winner_side_map(payload)

    cols = set(kills.columns)
    required = {"round_num", "attacker_steamid", "attacker_side"}
    if not required.issubset(cols):
        return attempts, wins

    tick_col = detect_column(kills, ["tick", "game_tick", "event_tick"])
    order_cols = ["round_num"] + ([tick_col] if tick_col else [])

    kill_rows = kills.sort(order_cols).select([
        c for c in [
            "round_num", tick_col,
            "attacker_steamid", "attacker_side",
            "victim_steamid", "victim_side",
        ] if c is not None and c in cols
    ]).to_dicts()

    first_kill_per_round = {}
    for row in kill_rows:
        rn = safe_int(row.get("round_num"), default=0)
        if rn <= 0 or rn in first_kill_per_round:
            continue

        atk_sid = normalize_player_id(row.get("attacker_steamid"))
        atk_side = normalize_side(row.get("attacker_side"))
        vic_sid = normalize_player_id(row.get("victim_steamid"))
        vic_side = normalize_side(row.get("victim_side"))

        if atk_sid is None or atk_side is None:
            continue
        if vic_sid is not None and vic_side is not None and atk_side == vic_side:
            continue

        first_kill_per_round[rn] = (atk_sid, atk_side)

    for rn, (atk_sid, atk_side) in first_kill_per_round.items():
        attempts[atk_sid] += 1
        if winner_side_by_round.get(rn) == atk_side:
            wins[atk_sid] += 1

    return attempts, wins


def compute_clutch_stats(payload):
    attempts = Counter()
    wins = Counter()

    ticks = to_polars(payload.get("ticks"))
    if ticks is None or ticks.height == 0:
        return attempts, wins

    cols = set(ticks.columns)
    required = {"round_num", "tick", "steamid", "side", "health"}
    if not required.issubset(cols):
        return attempts, wins

    alive = (
        ticks
        .select(["round_num", "tick", "steamid", "side", "health"])
        .with_columns([
            pl.col("round_num").cast(pl.Int64, strict=False).alias("round_num"),
            pl.col("tick").cast(pl.Int64, strict=False).alias("tick"),
            pl.col("steamid").cast(pl.Int64, strict=False).alias("steamid"),
            pl.col("side").cast(pl.Utf8, strict=False).alias("side"),
            pl.col("health").cast(pl.Int64, strict=False).alias("health"),
        ])
        .filter(pl.col("health") > 0)
    )

    if alive.height == 0:
        return attempts, wins

    alive_rows = alive.select(["round_num", "tick", "steamid", "side"]).to_dicts()

    per_round_tick_side_alive = defaultdict(lambda: defaultdict(lambda: {"CT": set(), "T": set()}))
    for row in alive_rows:
        rn = safe_int(row.get("round_num"), default=0)
        tk = safe_int(row.get("tick"), default=0)
        sid = normalize_player_id(row.get("steamid"))
        side = normalize_side(row.get("side"))
        if rn <= 0 or tk <= 0 or sid is None or side is None:
            continue
        per_round_tick_side_alive[rn][tk][side].add(sid)

    attempted_rounds = defaultdict(set)
    for rn, tick_map in per_round_tick_side_alive.items():
        for tk in sorted(tick_map.keys()):
            ct_alive = tick_map[tk]["CT"]
            t_alive = tick_map[tk]["T"]

            if len(ct_alive) == 1 and len(t_alive) >= 1:
                sid = next(iter(ct_alive))
                attempted_rounds[sid].add(rn)
            if len(t_alive) == 1 and len(ct_alive) >= 1:
                sid = next(iter(t_alive))
                attempted_rounds[sid].add(rn)

    for sid, rounds_set in attempted_rounds.items():
        attempts[sid] += len(rounds_set)

    winner_side_by_round = get_round_winner_side_map(payload)

    for sid, rounds_set in attempted_rounds.items():
        for rn in rounds_set:
            # find side of this player in round rn from any alive tick snapshot
            player_side = None
            tick_map = per_round_tick_side_alive.get(rn, {})
            for tk in sorted(tick_map.keys()):
                if sid in tick_map[tk]["CT"]:
                    player_side = "CT"
                    break
                if sid in tick_map[tk]["T"]:
                    player_side = "T"
                    break

            if player_side is not None and winner_side_by_round.get(rn) == player_side:
                wins[sid] += 1

    return attempts, wins


def build_final_team_scoreboards(payload):
    round_side = collect_round_side_members(payload)
    if not round_side:
        return None, None, None

    round_assignment = infer_team_assignment(round_side)
    explicit_names = collect_explicit_round_side_names(payload)
    team_labels = infer_team_display_names(round_assignment, explicit_names)

    # player -> inferred stable team
    player_team_counts = defaultdict(Counter)
    for r, sides in round_side.items():
        ct_team = round_assignment[r]["CT"]
        t_team = round_assignment[r]["T"]
        for sid in sides["CT"]:
            player_team_counts[sid][ct_team] += 1
        for sid in sides["T"]:
            player_team_counts[sid][t_team] += 1

    player_team = {}
    for sid, counts in player_team_counts.items():
        player_team[sid] = counts.most_common(1)[0][0]

    rounds_played = build_player_rounds_lookup(payload)
    name_lookup = build_player_name_lookup(payload)

    player_stats = defaultdict(lambda: {
        "kills": 0,
        "deaths": 0,
        "assists": 0,
        "damage": 0,
        "utility_damage": 0,
        "headshot_kills": 0,
        "shots": 0,
        "hits": 0,
    })

    kills = to_polars(payload.get("kills"))
    if kills is not None and kills.height > 0:
        cols = set(kills.columns)
        hs_col = detect_column(kills, ["is_headshot", "headshot", "hs", "is_hs"])

        needed = [c for c in [
            "attacker_steamid", "attacker_side",
            "victim_steamid", "victim_side",
            "assister_steamid",
            hs_col,
        ] if c is not None and c in cols]

        for row in kills.select(needed).to_dicts():
            a = normalize_player_id(row.get("attacker_steamid"))
            v = normalize_player_id(row.get("victim_steamid"))
            s = normalize_player_id(row.get("assister_steamid"))

            atk_side = normalize_side(row.get("attacker_side"))
            vic_side = normalize_side(row.get("victim_side"))
            same_side = atk_side is not None and vic_side is not None and atk_side == vic_side

            if a is not None and not same_side:
                player_stats[a]["kills"] += 1
                if hs_col is not None:
                    is_hs = bool(row.get(hs_col))
                    if is_hs:
                        player_stats[a]["headshot_kills"] += 1

            if v is not None and not same_side:
                player_stats[v]["deaths"] += 1

            if s is not None:
                player_stats[s]["assists"] += 1

    damages = to_polars(payload.get("damages"))
    if damages is not None and damages.height > 0:
        dmg_col = get_damage_column(damages)
        cols = set(damages.columns)

        needed = [c for c in [
            "attacker_steamid",
            "attacker_side",
            "victim_side",
            dmg_col,
            "weapon", "weapon_name", "weapon_class", "weapon_type", "weapon_item",
        ] if c is not None and c in cols]

        for row in damages.select(needed).to_dicts():
            a = normalize_player_id(row.get("attacker_steamid"))
            if a is None:
                continue

            atk_side = normalize_side(row.get("attacker_side"))
            vic_side = normalize_side(row.get("victim_side"))
            if atk_side is not None and vic_side is not None and atk_side == vic_side:
                continue

            dmg = safe_int(row.get(dmg_col), default=0) if dmg_col else 0
            if dmg > 0:
                player_stats[a]["damage"] += dmg
                player_stats[a]["hits"] += 1
                if is_utility_damage_row(row):
                    player_stats[a]["utility_damage"] += dmg

    shots = to_polars(payload.get("shots"))
    if shots is not None and shots.height > 0:
        cols = set(shots.columns)
        if "player_steamid" in cols:
            for row in shots.select(["player_steamid"]).to_dicts():
                sid = normalize_player_id(row.get("player_steamid"))
                if sid is not None:
                    player_stats[sid]["shots"] += 1

    entry_attempts, entry_wins = compute_entry_stats(payload)
    clutch_attempts, clutch_wins = compute_clutch_stats(payload)

    rows = []
    all_player_ids = sorted(set(player_stats.keys()) | set(player_team.keys()))
    for sid in all_player_ids:
        team_key = player_team.get(sid)
        if not team_key:
            continue

        ps = player_stats[sid]
        k = ps["kills"]
        d = ps["deaths"]
        a = ps["assists"]
        dmg = ps["damage"]
        udmg = ps["utility_damage"]
        hs_kills = ps["headshot_kills"]
        shots_n = ps["shots"]
        hits_n = ps["hits"]

        n_rounds = rounds_played.get(sid, 0)
        kd = (k / d) if d > 0 else float(k)
        adr = (dmg / n_rounds) if n_rounds > 0 else 0.0
        hs_pct = (100.0 * hs_kills / k) if k > 0 else 0.0
        acc_pct = (100.0 * hits_n / shots_n) if shots_n > 0 else 0.0

        e_att = entry_attempts.get(sid, 0)
        e_win = entry_wins.get(sid, 0)
        c_att = clutch_attempts.get(sid, 0)
        c_win = clutch_wins.get(sid, 0)

        rows.append({
            "team_key": team_key,
            "team": team_labels.get(team_key, team_key),
            "steamid": sid,
            "Player": name_lookup.get(sid, str(sid)),
            "K": k,
            "D": d,
            "A": a,
            "K/D": round(kd, 2),
            "ADR": round(adr, 1),
            "HS%": f"{round(hs_pct)}%",
            "ACC%": f"{round(acc_pct)}%",
            "Entry": f"{e_win}/{e_att}",
            "Clutch": f"{c_win}/{c_att}",
            "UDMG": udmg,
            "DMG": dmg,
        })

    if not rows:
        return None, round_assignment, team_labels

    scoreboard_df = pl.DataFrame(rows)
    return scoreboard_df, round_assignment, team_labels


def show_scoreboard_for_payload(payload, match_id, map_number, filename):
    scoreboard_df, round_assignment, team_labels = build_final_team_scoreboards(payload)

    print("\n====================================================")
    print(f"match_id={match_id} | map_number={map_number} | {filename}")

    if scoreboard_df is None or round_assignment is None:
        print("No scoreboard could be built (missing round side/player data).")
        return None

    print("\n=== SIDE ASSIGNMENT BY ROUND (inferred stable teams) ===")
    for r in sorted(round_assignment.keys()):
        ct_team = team_labels.get(round_assignment[r]["CT"], round_assignment[r]["CT"])
        t_team = team_labels.get(round_assignment[r]["T"], round_assignment[r]["T"])
        print(f"Round {r:>2}: CT={ct_team} | T={t_team}")

    print("\n=== FINAL SCOREBOARD PER TEAM ===")
    ordered_cols = ["Player", "K", "D", "A", "K/D", "ADR", "HS%", "ACC%", "Entry", "Clutch", "UDMG", "DMG"]

    for team_name in sorted(scoreboard_df["team"].unique().to_list()):
        team_board = (
            scoreboard_df
            .filter(pl.col("team") == team_name)
            .sort(["K", "DMG"], descending=[True, True])
            .select(ordered_cols)
        )
        print(f"\n--- {team_name} ---")
        display(team_board)

    return {
        "scoreboard": scoreboard_df,
        "round_assignment": round_assignment,
        "team_labels": team_labels,
    }


def build_all_final_team_scoreboards(parsed_cache_dir):
    entries = demo_cache.list_cached_demos(parsed_cache_dir)
    if not entries:
        print(f"No cached demos found in {parsed_cache_dir}")
        return {}

    print(f"Building final team scoreboards for {len(entries)} cached parsed matches...")
    results = {}

    for entry in entries:
        match_id = entry.get("match_id")
        map_number = entry.get("map_number")
        filename = entry.get("filename", "unknown")

        payload_i = demo_cache.load_parsed_demo(
            cache_dir=parsed_cache_dir,
            match_id=match_id,
            map_number=map_number,
        )

        if payload_i is None:
            print(f"\n[WARN] Could not load payload for match_id={match_id}, map_number={map_number}")
            continue

        result = show_scoreboard_for_payload(payload_i, match_id, map_number, filename)
        if result is not None:
            results[(match_id, map_number)] = result

    print("\nDone.")
    print(f"Successful scoreboards: {len(results)}/{len(entries)}")
    return results


# Run for all parsed cache entries
all_final_team_scoreboards = build_all_final_team_scoreboards(PARSED_CACHE_DIR)

Building final team scoreboards for 5 cached parsed matches...
[00:25:54.128] [DEBUG] [CACHE] Loaded parsed demo match=-1735857110 map=1431889500

match_id=-1735857110 | map_number=1431889500 | 2026-03-27_21-10-25_match_-1735857110_map_1431889500_de_cbble.pkl

=== SIDE ASSIGNMENT BY ROUND (inferred stable teams) ===
Round  1: CT=TeamA | T=TeamB
Round  2: CT=TeamA | T=TeamB
Round  3: CT=TeamA | T=TeamB
Round  4: CT=TeamA | T=TeamB
Round  5: CT=TeamA | T=TeamB
Round  6: CT=TeamA | T=TeamB
Round  7: CT=TeamA | T=TeamB
Round  8: CT=TeamA | T=TeamB
Round  9: CT=TeamA | T=TeamB
Round 10: CT=TeamA | T=TeamB
Round 11: CT=TeamA | T=TeamB
Round 12: CT=TeamA | T=TeamB
Round 13: CT=TeamB | T=TeamA
Round 14: CT=TeamB | T=TeamA
Round 15: CT=TeamB | T=TeamA
Round 16: CT=TeamB | T=TeamA
Round 17: CT=TeamB | T=TeamA
Round 18: CT=TeamB | T=TeamA

=== FINAL SCOREBOARD PER TEAM ===

--- TeamA ---


Player,K,D,A,K/D,ADR,HS%,ACC%,Entry,Clutch,UDMG,DMG
str,i64,i64,i64,f64,f64,str,str,str,str,i64,i64
"""Captain Blake""",18,15,6,1.2,127.2,"""61%""","""38%""","""0/3""","""0/5""",201,2290
"""Sindilia""",16,15,4,1.07,84.1,"""56%""","""27%""","""0/0""","""0/5""",57,1514
"""Couchyyy1337""",12,17,3,0.71,65.9,"""50%""","""31%""","""0/2""","""0/2""",27,1187
"""Dottersack""",11,17,2,0.65,57.2,"""27%""","""18%""","""0/0""","""0/1""",0,1030
"""Soulcrusher""",4,18,4,0.22,37.2,"""25%""","""17%""","""0/1""","""0/3""",13,670



--- TeamB ---


Player,K,D,A,K/D,ADR,HS%,ACC%,Entry,Clutch,UDMG,DMG
str,i64,i64,i64,f64,f64,str,str,str,str,i64,i64
"""KeTaNoS""",26,10,2,2.6,130.0,"""62%""","""24%""","""0/4""","""0/3""",16,2340
"""ᗢsupp""",17,13,4,1.31,91.4,"""47%""","""13%""","""0/5""","""0/2""",0,1645
"""Tundra ツ""",15,14,8,1.07,111.0,"""60%""","""18%""","""0/2""","""0/3""",206,1998
"""Inay""",15,12,4,1.25,81.1,"""60%""","""23%""","""0/0""","""0/0""",0,1459
"""Shouzen""",9,12,4,0.75,54.7,"""11%""","""21%""","""0/1""","""0/0""",18,985


[00:25:56.081] [DEBUG] [CACHE] Loaded parsed demo match=-1160320022 map=906937082

match_id=-1160320022 | map_number=906937082 | 2026-03-27_20-33-52_match_-1160320022_map_906937082_de_dust2.pkl

=== SIDE ASSIGNMENT BY ROUND (inferred stable teams) ===
Round  1: CT=TeamA | T=TeamB
Round  2: CT=TeamA | T=TeamB
Round  3: CT=TeamA | T=TeamB
Round  4: CT=TeamA | T=TeamB
Round  5: CT=TeamA | T=TeamB
Round  6: CT=TeamA | T=TeamB
Round  7: CT=TeamA | T=TeamB
Round  8: CT=TeamA | T=TeamB
Round  9: CT=TeamA | T=TeamB
Round 10: CT=TeamA | T=TeamB
Round 11: CT=TeamA | T=TeamB
Round 12: CT=TeamA | T=TeamB
Round 13: CT=TeamB | T=TeamA
Round 14: CT=TeamB | T=TeamA
Round 15: CT=TeamB | T=TeamA
Round 16: CT=TeamB | T=TeamA
Round 17: CT=TeamB | T=TeamA

=== FINAL SCOREBOARD PER TEAM ===

--- TeamA ---


...

[00:26:02.322] [DEBUG] [CACHE] Loaded parsed demo match=1 map=0

match_id=1 | map_number=0 | 2026-03-19_21-55-45_match_1_map_0_de_warden.pkl

=== SIDE ASSIGNMENT BY ROUND (inferred stable teams) ===
Round  1: CT=TeamA | T=TeamB
Round  2: CT=TeamA | T=TeamB
Round  3: CT=TeamA | T=TeamB
Round  4: CT=TeamA | T=TeamB
Round  5: CT=TeamA | T=TeamB
Round  6: CT=TeamA | T=TeamB
Round  7: CT=TeamA | T=TeamB
Round  8: CT=TeamA | T=TeamB
Round  9: CT=TeamA | T=TeamB
Round 10: CT=TeamA | T=TeamB
Round 11: CT=TeamA | T=TeamB
Round 12: CT=TeamA | T=TeamB
Round 13: CT=TeamB | T=TeamA
Round 14: CT=TeamB | T=TeamA
Round 15: CT=TeamB | T=TeamA
Round 16: CT=TeamB | T=TeamA
Round 17: CT=TeamB | T=TeamA
Round 18: CT=TeamB | T=TeamA
Round 19: CT=TeamB | T=TeamA
Round 20: CT=TeamB | T=TeamA
Round 21: CT=TeamB | T=TeamA
Round 22: CT=TeamB | T=TeamA

=== FINAL SCOREBOARD PER TEAM ===

--- TeamA ---


Player,K,D,A,K/D,ADR,HS%,ACC%,Entry,Clutch,UDMG,DMG
str,i64,i64,i64,f64,f64,str,str,str,str,i64,i64
"""Sindilia""",25,19,4,1.32,121.1,"""40%""","""28%""","""0/5""","""0/3""",13,2665
"""Tundra ツ""",23,13,2,1.77,111.0,"""30%""","""25%""","""0/5""","""0/4""",90,2441
"""Dottersack""",9,17,2,0.53,46.3,"""56%""","""18%""","""0/1""","""0/3""",21,1018
"""Dude""",5,18,5,0.28,27.3,"""40%""","""20%""","""0/0""","""0/1""",0,600
"""Chicca —ฅ/ᐠ. ̫ .ᐟ\ฅ —""",4,18,4,0.22,43.4,"""75%""","""21%""","""0/0""","""0/3""",0,954



--- TeamB ---


Player,K,D,A,K/D,ADR,HS%,ACC%,Entry,Clutch,UDMG,DMG
str,i64,i64,i64,f64,f64,str,str,str,str,i64,i64
"""Captain Blake""",23,10,2,2.3,106.2,"""52%""","""19%""","""0/3""","""0/4""",171,2337
"""KeTaNoS""",22,14,3,1.57,104.3,"""27%""","""28%""","""0/3""","""0/1""",0,2295
"""Amü""",15,14,2,1.07,70.5,"""27%""","""18%""","""0/3""","""0/3""",137,1551
"""Couchyyy1337""",14,15,2,0.93,71.6,"""50%""","""19%""","""0/1""","""0/2""",25,1576
"""prinoX""",11,13,3,0.85,50.1,"""18%""","""18%""","""0/1""","""0/0""",7,1152



Done.
Successful scoreboards: 5/5


## Analytics (Movement-stats)

### building player stat dictionaries

In [10]:
# --- BUILD PLAYER STATS FOR SINGLE DEMO ---

def build_player_stats_single(data):
    # --- SOURCE TABLES ---
    kills = data["kills"]
    damages = data["damages"]
    shots = data["shots"]
    rounds = data["rounds"]
    players_df = data["player_round_totals"]

    if kills is None or damages is None or shots is None or players_df is None:
        return None

    # convert to polars if needed
    if not isinstance(kills, pl.DataFrame):
        kills = pl.from_pandas(kills)
    if not isinstance(damages, pl.DataFrame):
        damages = pl.from_pandas(damages)
    if not isinstance(shots, pl.DataFrame):
        shots = pl.from_pandas(shots)
    if not isinstance(players_df, pl.DataFrame):
        players_df = pl.from_pandas(players_df)
    if not isinstance(rounds, pl.DataFrame):
        rounds = pl.from_pandas(rounds)

    # --- PLAYER INDEX ---
    players = (
        players_df
        .filter(pl.col("side") == "all")
        .select(["steamid", "name"])
        .unique(subset=["steamid"])
    )

    # --- NORMALIZE EVENTS ---
    kills_norm = kills.select([
        pl.col("attacker_steamid").alias("steamid"),
        pl.lit(1).alias("kills"),
        pl.lit(0).alias("deaths"),
        pl.lit(0).alias("damage"),
        pl.lit(0).alias("shots"),
    ])

    deaths_norm = kills.select([
        pl.col("victim_steamid").alias("steamid"),
        pl.lit(0).alias("kills"),
        pl.lit(1).alias("deaths"),
        pl.lit(0).alias("damage"),
        pl.lit(0).alias("shots"),
    ])

    damage_norm = damages.select([
        pl.col("attacker_steamid").alias("steamid"),
        pl.lit(0).alias("kills"),
        pl.lit(0).alias("deaths"),
        pl.col("dmg_health_real").alias("damage"),
        pl.lit(0).alias("shots"),
    ])

    shots_norm = shots.select([
        pl.col("player_steamid").alias("steamid"),
        pl.lit(0).alias("kills"),
        pl.lit(0).alias("deaths"),
        pl.lit(0).alias("damage"),
        pl.lit(1).alias("shots"),
    ])

    all_events = pl.concat([
        kills_norm,
        deaths_norm,
        damage_norm,
        shots_norm
    ])

    # --- AGG ---
    stats = (
        all_events
        .group_by("steamid")
        .agg([
            pl.sum("kills"),
            pl.sum("deaths"),
            pl.sum("damage"),
            pl.sum("shots"),
        ])
    )

    # --- DERIVED ---
    n_rounds = max(1, rounds.height)

    stats = stats.with_columns([
        (pl.col("kills") / pl.when(pl.col("deaths") == 0).then(1).otherwise(pl.col("deaths"))).alias("kd"),
        (pl.col("damage") / n_rounds).alias("adr"),
        (pl.col("kills") / pl.when(pl.col("shots") == 0).then(1).otherwise(pl.col("shots"))).alias("accuracy")
    ])

    stats = stats.join(players, on="steamid", how="left")

    return stats

# --- MULTI DEMO WRAPPER ---

def build_player_stats_all(demo_data):
    print("[Stats] Building across all demos...\n")

    per_demo_stats = []

    for (match_id, map_number), data in demo_data.items():
        print(f"[Stats] match={match_id} map={map_number}")

        stats = build_player_stats_single(data)

        if stats is None:
            print("  -> skipped (missing data)")
            continue

        stats = stats.with_columns([
            pl.lit(match_id).alias("match_id"),
            pl.lit(map_number).alias("map_number")
        ])

        per_demo_stats.append(stats)

    if not per_demo_stats:
        print("[Stats] No valid demos")
        return None, None

    # --- CONCAT ALL PER-DEMO STATS ---
    all_stats = pl.concat(per_demo_stats)

    # --- BUILD ONE STATBOARD PER MATCH ---
    match_statboards = {}
    match_ids = sorted(all_stats["match_id"].unique().to_list())

    for match_id in match_ids:
        match_stats = all_stats.filter(pl.col("match_id") == match_id)

        statboard = (
            match_stats
            .group_by("steamid")
            .agg([
                pl.first("name"),
                pl.n_unique("map_number").alias("maps_played"),
                pl.sum("kills"),
                pl.sum("deaths"),
                pl.sum("damage"),
                pl.sum("shots"),
            ])
            .with_columns([
                (pl.col("kills") / pl.when(pl.col("deaths") == 0).then(1).otherwise(pl.col("deaths"))).alias("kd"),
                (pl.col("damage") / pl.when(pl.col("maps_played") == 0).then(1).otherwise(pl.col("maps_played"))).alias("adr"),
                (pl.col("kills") / pl.when(pl.col("shots") == 0).then(1).otherwise(pl.col("shots"))).alias("accuracy"),
            ])
            .sort("kills", descending=True)
        )

        match_statboards[match_id] = statboard

    print("\n[Stats] Per-match statboards\n")
    for match_id in sorted(match_statboards.keys()):
        print(f"=== MATCH {match_id} ===")
        display(match_statboards[match_id])

    return match_statboards, all_stats


# --- RUN ---

player_stats_by_match, player_stats_per_demo = build_player_stats_all(demo_data)
# legacy compatibility: no single global board anymore
player_stats_global = None

[Stats] Building across all demos...

[Stats] match=1 map=0
[Stats] match=3 map=0
[Stats] match=4 map=0
[Stats] match=-1160320022 map=906937082
[Stats] match=-1735857110 map=1431889500

[Stats] Per-match statboards

=== MATCH -1735857110 ===


steamid,name,maps_played,kills,deaths,damage,shots,kd,adr,accuracy
u64,str,u32,i32,i32,i32,i32,f64,f64,f64
76561198045998047,"""KeTaNoS""",1,26,10,2380,278,2.6,2380.0,0.093525
76561198051680113,"""Captain Blake""",1,18,15,2290,259,1.2,2290.0,0.069498
76561198158369257,"""ᗢsupp""",1,17,13,1663,263,1.307692,1663.0,0.064639
76561198095760902,"""Sindilia""",1,16,15,1529,229,1.066667,1529.0,0.069869
76561198007257679,"""Inay""",1,15,12,1459,205,1.25,1459.0,0.073171
…,…,…,…,…,…,…,…,…,…
76561198034161077,"""Couchyyy1337""",1,12,17,1187,175,0.705882,1187.0,0.068571
76561198240903150,"""Dottersack""",1,11,17,1030,269,0.647059,1030.0,0.040892
76561198028953143,"""Shouzen""",1,9,12,1005,245,0.75,1005.0,0.036735


=== MATCH -1160320022 ===


steamid,name,maps_played,kills,deaths,damage,shots,kd,adr,accuracy
u64,str,u32,i32,i32,i32,i32,f64,f64,f64
76561198045998047,"""KeTaNoS""",1,25,10,2217,291,2.5,2217.0,0.085911
76561198034161077,"""Couchyyy1337""",1,23,12,2175,350,1.916667,2175.0,0.065714
76561198158369257,"""ᗢsupp""",1,16,13,1647,226,1.230769,1647.0,0.070796
76561198354578034,"""Soulcrusher""",1,15,9,1249,226,1.666667,1249.0,0.066372
76561198051680113,"""Captain Blake""",1,12,16,1267,170,0.75,1267.0,0.070588
…,…,…,…,…,…,…,…,…,…
76561197987271352,"""Tundra ツ""",1,8,10,1664,307,0.8,1664.0,0.026059
76561198095760902,"""Sindilia""",1,8,17,908,132,0.470588,908.0,0.060606
76561198240903150,"""Dottersack""",1,7,12,995,120,0.583333,995.0,0.058333


=== MATCH 1 ===


steamid,name,maps_played,kills,deaths,damage,shots,kd,adr,accuracy
u64,str,u32,i32,i32,i32,i32,f64,f64,f64
76561198095760902,"""Sindilia""",1,25,19,2677,341,1.315789,2677.0,0.073314
76561197987271352,"""Tundra ツ""",1,23,13,2441,227,1.769231,2441.0,0.101322
76561198051680113,"""Captain Blake""",1,23,10,2337,445,2.3,2337.0,0.051685
76561198045998047,"""KeTaNoS""",1,22,14,2295,215,1.571429,2295.0,0.102326
76561198962927846,"""Amü""",1,16,15,1580,377,1.066667,1580.0,0.04244
76561198034161077,"""Couchyyy1337""",1,14,15,1576,219,0.933333,1576.0,0.063927
76561198049175684,"""prinoX""",1,12,14,1182,258,0.857143,1182.0,0.046512
76561198240903150,"""Dottersack""",1,9,17,1018,213,0.529412,1018.0,0.042254
76561197970397706,"""Dude""",1,5,18,600,167,0.277778,600.0,0.02994


=== MATCH 3 ===


steamid,name,maps_played,kills,deaths,damage,shots,kd,adr,accuracy
u64,str,u32,i32,i32,i32,i32,f64,f64,f64
76561198004039473,"""blu""",1,31,15,2996,466,2.066667,2996.0,0.066524
76561198045998047,"""KeTaNoS""",1,29,16,2666,382,1.8125,2666.0,0.075916
76561197987271352,"""Tundra ツ""",1,21,15,1877,427,1.4,1877.0,0.04918
76561198095760902,"""Sindilia""",1,20,19,2613,309,1.052632,2613.0,0.064725
76561198051680113,"""Captain Blake""",1,16,20,1556,264,0.8,1556.0,0.060606
…,…,…,…,…,…,…,…,…,…
76561198034161077,"""Couchyyy1337""",1,14,20,1957,419,0.7,1957.0,0.033413
76561198354578034,"""Soulcrusher""",1,14,18,1355,375,0.777778,1355.0,0.037333
76561198010796900,"""-Zuria-""",1,10,21,1384,408,0.47619,1384.0,0.02451


=== MATCH 4 ===


steamid,name,maps_played,kills,deaths,damage,shots,kd,adr,accuracy
u64,str,u32,i32,i32,i32,i32,f64,f64,f64
76561198051680113,"""Captain Blake""",1,21,7,1820,347,3.0,1820.0,0.060519
76561197987271352,"""Tundra ツ""",1,13,7,1442,370,1.857143,1442.0,0.035135
76561198045998047,"""KeTaNoS""",1,13,9,1222,141,1.444444,1222.0,0.092199
76561197994930572,"""◄TEAM.sl1χχ | єχχтяємє""",1,12,12,1210,296,1.0,1210.0,0.040541
76561198095760902,"""Sindilia""",1,10,13,1113,163,0.769231,1113.0,0.06135
76561197970397706,"""Dude""",1,10,6,959,136,1.666667,959.0,0.073529
76561198004039473,"""blu""",1,8,12,922,248,0.666667,922.0,0.032258
76561198240903150,"""Dottersack""",1,6,9,1007,363,0.666667,1007.0,0.016529
76561198034161077,"""Couchyyy1337""",1,5,13,634,358,0.384615,634.0,0.013966


### Filter ticks to create a dict of tickstamps of live-start to live-end (excluding freeze-time)

In [ ]:
# --- SINGLE DEMO ---

def build_round_windows_single(data):
    rounds = data.get("rounds")

    if rounds is None:
        return None

    if not isinstance(rounds, pl.DataFrame):
        rounds = pl.from_pandas(rounds)

    round_windows = (
        rounds
        .select([
            "round_num",
            pl.col("freeze_end").alias("live_start"),
            pl.col("end").alias("live_end"),
        ])
        .sort("round_num")
    )

    return round_windows

# --- MULTI DEMO ---

def build_round_windows_all(demo_data):
    print("[Windows] Building round windows across all demos...\n")

    results = {}

    for (match_id, map_number), data in demo_data.items():
        print(f"[Windows] match={match_id} map={map_number}")

        windows = build_round_windows_single(data)

        if windows is None:
            print("  -> skipped (no rounds)")
            continue

        results[(match_id, map_number)] = windows

    print("\n[Windows] Done")
    return results

# --- RUN ---

round_windows_all = build_round_windows_all(demo_data)

### Get movement data within the filtered frame per round

In [ ]:



# extract tickrate from demo
def get_tickrate(data):
    header = data.get("header")

    if header is None:
        raise ValueError("Missing demo header")

    # awpy usually provides this
    tickrate = header.get("tickrate")

    if tickrate is None:
        # fallback (common CS2 default)
        tickrate = 128

    return float(tickrate)


# single demo
def build_movement_stats_single(data, live_round_windows):
    print("[Movement] Building...")

    tickrate = get_tickrate(data)
    print(f"[Movement] Tickrate: {tickrate}")

    ticks = data["ticks"]
    players_df = data["player_round_totals"]

    # players
    players = (
        players_df
        .filter(pl.col("side") == "all")
        .select(["steamid", "name"])
        .unique(subset=["steamid"])
    )

    # join windows
    ticks = ticks.join(
        live_round_windows,
        on="round_num",
        how="left"
    )

    # sort
    ticks = ticks.sort(["steamid", "tick"])

    # deltas
    ticks = ticks.with_columns([
        pl.col("X").diff().over("steamid").alias("dx"),
        pl.col("Y").diff().over("steamid").alias("dy"),
        pl.col("Z").diff().over("steamid").alias("dz"),
        pl.col("tick").diff().over("steamid").alias("dt"),
    ])

    # distance
    ticks = ticks.with_columns([
        ((pl.col("dx")**2 + pl.col("dy")**2 + pl.col("dz")**2).sqrt()).alias("distance")
    ])

    # speed
    ticks = ticks.with_columns([
        (pl.col("distance") / pl.col("dt").fill_null(1)).alias("units_per_tick")
    ])

    ticks = ticks.with_columns([
        (pl.col("units_per_tick") * tickrate).alias("speed")
    ])

    # live window
    ticks = ticks.with_columns([
        (
            (pl.col("tick") >= pl.col("live_start")) &
            (pl.col("tick") <= pl.col("live_end"))
        ).alias("in_live")
    ])

    # filtered metrics
    ticks = ticks.with_columns([
        pl.when((pl.col("health") > 0) & pl.col("in_live"))
        .then(pl.col("distance"))
        .otherwise(0)
        .alias("distance_alive"),

        pl.when(
            (pl.col("health") > 0) &
            pl.col("in_live") &
            (pl.col("speed") < 400)
        )
        .then(pl.col("speed"))
        .otherwise(None)
        .alias("speed_valid"),

        pl.when((pl.col("health") > 0) & pl.col("in_live"))
        .then(1)
        .otherwise(0)
        .alias("alive_tick")
    ])

    # aggregate
    movement = (
        ticks
        .group_by("steamid")
        .agg([
            pl.sum("distance_alive").alias("total_distance_units"),
            pl.max("speed_valid").alias("max_speed_units_s"),
            pl.sum("alive_tick").alias("ticks_alive")
        ])
    )

    n_rounds = live_round_windows.height

    movement = movement.with_columns([
        (pl.col("ticks_alive") / tickrate).alias("alive_seconds"),

        (
            pl.col("total_distance_units") /
            (pl.col("ticks_alive") / tickrate)
        ).alias("avg_speed_units_s"),

        (pl.col("total_distance_units") / n_rounds).alias("distance_per_round_units"),

        (pl.col("total_distance_units") * 0.0254).alias("total_distance_m"),

        (
            (pl.col("total_distance_units") /
             (pl.col("ticks_alive") / tickrate)) * 0.0254
        ).alias("avg_speed_m_s"),
    ])

    movement = movement.join(players, on="steamid", how="left")

    movement = movement.select([
        "steamid",
        "name",
        "total_distance_units",
        "total_distance_m",
        "distance_per_round_units",
        "avg_speed_units_s",
        "avg_speed_m_s",
        "max_speed_units_s",
        "ticks_alive",
        "alive_seconds",
    ])

    movement = movement.fill_null(0)
    movement = movement.sort("total_distance_units", descending=True)

    print("[Movement] Done")

    return movement, ticks, players


# multi demo
def build_movement_stats_all(demo_data, round_windows_all):
    print("[Movement] Processing all demos...\n")

    results = {}

    for key in demo_data:
        data = demo_data[key]
        windows = round_windows_all.get(key)

        if windows is None:
            continue

        print(f"[Movement] match={key[0]} map={key[1]}")

        movement, ticks, players = build_movement_stats_single(data, windows)

        results[key] = {
            "movement": movement,
            "ticks": ticks,
            "players": players
        }

    print("\n[Movement] Done")

    return results


# run
movement_data = build_movement_stats_all(demo_data, round_windows_all)


### Render a plot for showing some stuff

In [ ]:
import matplotlib.pyplot as plt
import math
import polars as pl
import matplotlib.cm as cm

BIN_TICKS = 32
BIN_SIZE = None  # will be set dynamically


def plot_speed_per_round(data, ticks, players, live_round_windows, player_filter=None):
    print("[Plot] Building ROUND-SPLIT curves...")

    tickrate = get_tickrate(data)
    print(f"[Plot] Tickrate: {tickrate}")

    global BIN_SIZE
    BIN_SIZE = BIN_TICKS / tickrate

    # filter players
    if player_filter is not None:
        if not isinstance(player_filter, list):
            player_filter = [player_filter]

        if all(isinstance(p, str) for p in player_filter):
            players = players.filter(pl.col("name").is_in(player_filter))
        else:
            players = players.filter(pl.col("steamid").is_in(player_filter))

        selected_ids = players["steamid"].to_list()
        ticks = ticks.filter(pl.col("steamid").is_in(selected_ids))

        print(f"[Plot] Filtering: {players['name'].to_list()}")

    # join windows
    ticks_plot = ticks.join(
        live_round_windows,
        on="round_num",
        how="left"
    )

    # round time
    ticks_plot = ticks_plot.with_columns([
        ((pl.col("tick") - pl.col("live_start")) / tickrate).alias("round_time_sec")
    ])

    # live filter
    ticks_plot = ticks_plot.with_columns([
        (
            (pl.col("tick") >= pl.col("live_start")) &
            (pl.col("tick") <= pl.col("live_end"))
        ).alias("in_live")
    ]).filter(pl.col("in_live"))

    # speed
    ticks_plot = ticks_plot.with_columns([
        pl.when(
            (pl.col("dt") <= 2) &
            (pl.col("health") > 0)
        )
        .then(pl.col("speed") * 0.0254)
        .otherwise(None)
        .alias("speed_m_s")
    ])

    # player map
    player_dicts = players.to_dicts()
    player_map = {p["steamid"]: p["name"] for p in player_dicts}

    # colors
    cmap = cm.get_cmap("tab10", len(player_dicts))
    color_map = {
        p["steamid"]: cmap(i)
        for i, p in enumerate(player_dicts)
    }

    # mode
    use_raw = players.height == 0

    if use_raw:
        print("[Plot] RAW")
        agg = (
            ticks_plot
            .select(["round_num", "steamid", "round_time_sec", "speed_m_s"])
            .sort(["round_num", "steamid", "round_time_sec"])
        )
    else:
        print("[Plot] SMOOTHED")

        ticks_plot = ticks_plot.with_columns([
            (pl.col("round_time_sec") // BIN_SIZE).alias("time_bin")
        ])

        agg = (
            ticks_plot
            .group_by(["round_num", "steamid", "time_bin"])
            .agg(pl.median("speed_m_s").alias("speed_m_s"))
            .sort(["round_num", "steamid", "time_bin"])
        )

    # layout
    rounds = agg["round_num"].unique().to_list()
    cols = 4
    rows = math.ceil(len(rounds) / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 3), dpi=200)
    axes = axes.flatten()

    # loop rounds
    for i, r in enumerate(rounds):
        ax = axes[i]
        df_r = agg.filter(pl.col("round_num") == r)

        # stats (shared)
        stats = (
            ticks_plot
            .filter(
                (pl.col("round_num") == r) &
                (pl.col("health") > 0) &
                (pl.col("dt") <= 2)
            )
            .group_by("steamid")
            .agg([
                pl.mean("speed_m_s").alias("avg_speed"),
                (pl.count() / tickrate).alias("time_alive_sec")
            ])
        )

        # plot lines
        for p in player_dicts:
            sid = p["steamid"]
            name = p["name"]

            df_p = df_r.filter(pl.col("steamid") == sid)
            if df_p.height == 0:
                continue

            if use_raw:
                time_sec = df_p["round_time_sec"].to_list()
            else:
                time_sec = (df_p["time_bin"] * BIN_SIZE).to_list()

            speed = df_p["speed_m_s"].to_list()

            ax.plot(
                time_sec,
                speed,
                linewidth=1.2,
                alpha=0.9,
                color=color_map[sid],
                label=name
            )

        # labels / info
        if stats.height > 0:
            if players.height == 1:
                row = stats.to_dicts()[0]
                name = player_map.get(row["steamid"], "?")

                ax.text(
                    0.02, 0.02,
                    f"{name}\n{row['avg_speed']:.1f} m/s\n{row['time_alive_sec']:.1f}s alive",
                    transform=ax.transAxes,
                    fontsize=9,
                    va="bottom",
                    bbox=dict(boxstyle="round", alpha=0.2)
                )
            else:
                lines = []
                for row in stats.to_dicts():
                    name = player_map.get(row["steamid"], "?")
                    lines.append(
                        f"{name}: {row['avg_speed']:.1f} m/s | {row['time_alive_sec']:.1f}s"
                    )

                ax.text(
                    0.98, 0.98,
                    "\n".join(lines),
                    transform=ax.transAxes,
                    fontsize=7,
                    va="top",
                    ha="right",
                    bbox=dict(boxstyle="round", alpha=0.2)
                )

                ax.legend(fontsize=7, loc="lower left")

        # axis
        max_time = (
            ticks_plot
            .filter(pl.col("round_num") == r)
            .select(pl.col("round_time_sec").max())
            .item()
        )

        ax.set_xlim(0, min(120, max_time + 1))
        ax.set_ylim(0, 15)

        ax.set_title(f"Round {r}", fontsize=10)
        ax.set_xlabel("Time (s)", fontsize=8)
        ax.set_ylabel("Speed", fontsize=8)
        ax.grid(alpha=0.2)

        # shift line
        SHIFT_SPEED = 5.95
        ax.axhline(y=SHIFT_SPEED, linestyle="--", linewidth=1.0, alpha=0.6, color="black")

    # cleanup
    for j in range(i + 1, len(axes)):
        fig.delaxes(axes[j])

    # --- GLOBAL TITLE ---

    header = data.get("header", {})
    map_name = header.get("map_name", "unknown")

    n_rounds = len(rounds)

    # player list
    player_names = players["name"].to_list()
    player_names_sorted = sorted(player_names)

    players_text = ", ".join(player_names_sorted)

    title_text = f"{map_name} | {n_rounds} rounds"
    subtitle_text = f"Players: {players_text}"

    fig.suptitle(
        title_text,
        fontsize=16,
        y=1.04
    )

    fig.text(
        0.5, 1.01,
        subtitle_text,
        ha="center",
        fontsize=10
    )

    plt.tight_layout()
    plt.show()


def plot_game_for_player(key, player_filter):
    data = demo_data[key]
    ticks = movement_data[key]["ticks"]
    players = movement_data[key]["players"]
    windows = round_windows_all[key]

    plot_speed_per_round(
        data,
        ticks,
        players,
        windows,
        player_filter=player_filter
    )

# call

# example keys
game_1 = (1, 0)
game_3 = (3, 0)
game_4 = (4, 0)
# example player filters <steamids>
Dottersack = 76561198240903150
Tundra = 76561197987271352
captain_blake = 76561198051680113
Ketanos = 76561198045998047

plot_game_for_player(game_1, player_filter=Ketanos)
plot_game_for_player(game_3, player_filter=[Dottersack, Tundra, captain_blake])

### pie charts to display movement status per round

In [ ]:
import numpy as np

def plot_speed_pies(data, ticks, players, live_round_windows, player_filter=None):
    print("[Pie] Building distributions...")

    tickrate = get_tickrate(data)

    # filter players
    if player_filter is not None:
        if not isinstance(player_filter, list):
            player_filter = [player_filter]

        if all(isinstance(p, str) for p in player_filter):
            players = players.filter(pl.col("name").is_in(player_filter))
        else:
            players = players.filter(pl.col("steamid").is_in(player_filter))

        if players.height == 0:
            print("[Pie] No matching players")
            return

        selected_ids = players["steamid"].to_list()
        ticks = ticks.filter(pl.col("steamid").is_in(selected_ids))

        print(f"[Pie] Filtering: {players['name'].to_list()}")

    # join round windows
    ticks_plot = ticks.join(
        live_round_windows,
        on="round_num",
        how="left"
    )

    # live flag
    ticks_plot = ticks_plot.with_columns([
        (
            (pl.col("tick") >= pl.col("live_start")) &
            (pl.col("tick") <= pl.col("live_end"))
        ).alias("in_live")
    ])

    # filter valid ticks
    ticks_plot = ticks_plot.filter(
        (pl.col("in_live")) &
        (pl.col("health") > 0) &
        (pl.col("dt") <= 2)
    )

    if ticks_plot.height == 0:
        print("[Pie] No valid ticks after filtering")
        return

    # convert to m/s
    ticks_plot = ticks_plot.with_columns([
        (pl.col("speed") * 0.0254).alias("speed_m_s")
    ])

    # thresholds
    STAND = 0.3
    SNEAK = 5.95

    # classify movement
    ticks_plot = ticks_plot.with_columns([
        pl.when(pl.col("speed_m_s") < STAND)
        .then(pl.lit("Stand"))
        .when(pl.col("speed_m_s") < SNEAK)
        .then(pl.lit("Sneak"))
        .when(pl.col("speed_m_s") <= 13.0)
        .then(pl.lit("Run"))
        .otherwise(pl.lit("Fly"))
        .alias("movement")
    ])

    # time per tick
    ticks_plot = ticks_plot.with_columns([
        pl.lit(1.0 / tickrate).alias("tick_time")
    ])

    # aggregate
    agg_game = (
        ticks_plot
        .group_by(["steamid", "movement"])
        .agg(pl.col("tick_time").sum().alias("time"))
    )

    # plotting
    player_dicts = players.to_dicts()
    categories = ["Stand", "Sneak", "Run", "Fly"]

    print("[Pie] Plotting game-level...")

    n_players = len(player_dicts)
    cols = min(4, n_players)
    rows = math.ceil(n_players / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 4), dpi=150)

    # normalize axes safely
    if isinstance(axes, np.ndarray):
        axes = axes.flatten()
    else:
        axes = [axes]

    for i, p in enumerate(player_dicts):
        sid = p["steamid"]
        name = p["name"]

        ax = axes[i]
        df_p = agg_game.filter(pl.col("steamid") == sid)

        values = []
        for cat in categories:
            row = df_p.filter(pl.col("movement") == cat)
            values.append(row["time"].item() if row.height > 0 else 0.0)

        if sum(values) == 0:
            ax.set_title(name)
            ax.text(0.5, 0.5, "No data", ha="center", va="center")
            ax.axis("off")
            continue

        ax.pie(values, labels=categories, autopct="%1.1f%%", startangle=90)
        ax.set_title(name)

    # remove unused axes
    for j in range(len(player_dicts), len(axes)):
        fig.delaxes(axes[j])

    header = data.get("header", {})
    map_name = header.get("map_name", "unknown")

    fig.suptitle(f"{map_name} | Movement (Game)", fontsize=14)
    
    plt.tight_layout()
    plt.show()


# wrapper (IMPORTANT — fixes your earlier mistake)
def plot_game_pies(key, player_filter=None):
    data = demo_data[key]
    ticks = movement_data[key]["ticks"]
    players = movement_data[key]["players"]
    windows = round_windows_all[key]

    plot_speed_pies(
        data,
        ticks,
        players,
        windows,
        player_filter=player_filter
    )


# examples (correct usage)
plot_game_pies(game_1, player_filter=[Dottersack, Tundra, captain_blake, Ketanos])
plot_game_pies(game_3, player_filter=[Dottersack, Tundra, captain_blake, Ketanos])
plot_game_pies(game_4, player_filter=[Dottersack, Tundra, captain_blake, Ketanos])